### RELATÓRIO - LH NAUTICAL ⚓

#### 1 - EDA

In [38]:
# IMPORTANDO BIBLIOTECAS
import pandas as pd 
from datetime import datetime

# CARREGANDO O DATASET
df = pd.read_csv('../data/raw/vendas_2023_2024.csv')

1.1 - Visão geral do dataset (quantidade de linhas, colunas e intervalo de datas):

In [39]:
#LINHAS E COLUNAS
print(f"Linhas: {df.shape[0]}\nColunas: {df.shape[1]}")

# CONFERINDO AS DATAS
df_data = df.copy()
df_data['sale_date'] = pd.to_datetime(df['sale_date'], format='mixed', dayfirst=True)
print(f"Data mínima: {df_data['sale_date'].min().strftime('%d-%m-%y')} e Data máxima: {df_data['sale_date'].max().strftime('%d-%m-%y')}")

Linhas: 9895
Colunas: 6
Data mínima: 01-01-23 e Data máxima: 31-12-24


1.2 - Análise de valores numéricos da coluna "total":

In [40]:
#ANÁLISE DA COLUNA TOTAL
print(f"Valor mínimo: {df['total'].min()}")
print(f"Valor máximo: {df['total'].max()}")
print(f"Valor médio: {round(df['total'].mean(), 2)}")
print(f"Mediana: {df['total'].median()}")

#AVALIANDO OUTLIERS PELO MÉTODO INTERQUARTIL
q1 = df['total'].quantile(0.25) 
q3 = df['total'].quantile(0.75)
iqr = q3 - q1
lim_inferior = q1 - 1.5 * iqr
lim_superior = q3 + 1.5 * iqr

outliers_inferiores = df[df['total'] < lim_inferior]
outliers_superiores = df[df['total'] > lim_superior]

print(f"Q1: {q1}\nQ3: {q3}")
print(f"limite inferior: {lim_inferior}\nlimite superior: {lim_superior}")
print(f"outliers inferiores: {outliers_inferiores.shape[0]}")
print(f"outliers superiores: {outliers_superiores.shape[0]}")

# RANKING DE OUTLIERS
outliers_ranking = outliers_superiores.sort_values(by='total', ascending=False)
top_outliers = outliers_ranking[['id_product', 'total']].head(10)
print("Top 10 maiores outliers de vendas:")
print(top_outliers)

Valor mínimo: 294.5
Valor máximo: 2222973.0
Valor médio: 263797.83
Mediana: 82225.0
Q1: 23138.2
Q3: 339094.5
limite inferior: -450796.24999999994
limite superior: 813028.95
outliers inferiores: 0
outliers superiores: 1018
Top 10 maiores outliers de vendas:
      id_product       total
8905          76  2222973.00
3873          76  2222973.00
9623          76  2222973.00
1578          76  2222973.00
7930          73  2147399.00
8856          76  2111824.35
1974          76  2111824.35
8800          76  2074775.00
5014          76  2074775.00
5183          97  2030026.00


Os outliers significativos são os superiores com 1018 casos (cerca de 10% dos dados), que indicam valores  elevados acima dos 2 milhões, principalmente quandoo comparados às medidas de tendencia central deste dataframe. A média de 263 mil e a mediana 82 mil evidenciam a influência dos extremos de cima.

Além disso, só de observar o ranking dos 10 maiores outliers, percebemos mesmo produto (ID 76) aparecendo multiplas vezes entre os maiores, sendo que em quatro delas com o mesmo valor máximo de 2222973, o que pode nos indicar alguma anomalia, como talvez, registros duplicados.


In [41]:
#ANALISANDO OUTLIERS SUSPEITOS
outliers_suspeitos = df[(df['id_product'] == 76) & (df['total'] > 222900)]
print(outliers_suspeitos.head())

      id  id_client  id_product  qtd       total   sale_date
175  179         41          76   14  1971036.25  2024-09-27
598  608         23          76    4   592793.00  29-11-2023
671  681         32          76    3   422365.25  2024-10-06
737  747         11          76   12  1778379.00  2024-02-13
763  773         36          76   10  1407882.90  04-04-2024


Apesar da possibilidade de comportamento anômalo, com o mesmo produto registrando o mesmo total elevado quatro vezes, podemos visualizar que não se trata de duplicatas:

São apenas compras do mesmo produto em mesma quantidade, realizadas por clientes diferentes em datas distintas.

1.3 - Interpretação e diagnóstico para análises futuras:

In [43]:
#ANALISANDO VALORES NULOS, NEGATIVOS E DUPLICADOS
valores_nulos = df.isnull().sum()
print(f"Valores nulos:\n{valores_nulos}")
valores_negativos = (df.select_dtypes(include='number') < 0).sum()
print(f"Valores negativos:\n{valores_negativos}")
valores_duplicados = df.duplicated().sum()
print(f"Valores duplicados:\n {valores_duplicados}")

Valores nulos:
id            0
id_client     0
id_product    0
qtd           0
total         0
sale_date     0
dtype: int64
Valores negativos:
id            0
id_client     0
id_product    0
qtd           0
total         0
dtype: int64
Valores duplicados:
 0


In [44]:
#ANALISANDO FORMATOS DA COLUNA sale_date
df['sale_date'].astype(str).str.replace(r'\d', 'X', regex=True).value_counts()

sale_date
XX-XX-XXXX    4982
XXXX-XX-XX    4913
Name: count, dtype: int64

##### Conclusão:
Os dados são de qualidade, não constando valores nulos, negativos ou duplicados. Entretanto, temos inconsistências no formato padrão de datas da coluna "sale_date", que apresenta dois tipos de configuração. 

Ademais, podemos considerar o dataset parcialmente apto para prosseguir nas análises, e que estará completamente pronto após solucionadas inconsistências.

#### 2 - PRODUTOS


In [45]:
#IMPORTANDO BIBLIOTECA
import pandas as pd

# CARREGANDO O DATASET
df = pd.read_csv('../data/raw/produtos_raw.csv')

2.1 - Padronização dos nomes das categorias de produtos (eletrônicos, propulsão e ancoragem):

In [46]:
# APONTANDO AS CATEGORIAS ÚNICAS 
print(df['actual_category'].unique())

# FUNCÃO PARA NORMALIZAR AS CATEGORIAS DE PRODUTOS
def normalizar_categoria(categoria):
    '''Normaliza a categoria de produtos: retira os espaços, converte para minúsculas e categoriza com base em radicais em comum'''
    categoria = categoria.strip().lower().replace(' ', '')
    if 'el' in categoria:
        return 'eletrônicos'
    elif 'pr' in categoria:
        return 'propulsão'
    elif 'anc' in categoria or 'cor' in categoria:
        return 'ancoragem'
    return categoria

# APLICANDO A FUNÇÃO DE NORMALIZAÇÃO NA COLUNA 'actual_category'
df['actual_category'] = df['actual_category'].apply(normalizar_categoria)

# CONFERINDO RESULTADOS DA NORMALIZAÇÃO
print(f"{df['actual_category'].value_counts()}\n Total de registros: {df['actual_category'].value_counts().sum()}")

<StringArray>
[          'ELETRONICOS', 'E L E T R Ô N I C O S',           'Eletrunicos',
           'Eletronicoz',           'eLeTrÔnIcOs',           'eletrônicos',
           'Eletrônicos',          'Eletroniscos',           'Eletronicos',
           'eletronicos',           'EletrônicoS',           'ELEtRÔNICOS',
             'PROPULSAO',             'Propulção',                  'Prop',
            'Propulssão',             'propulsao',     'P R O P U L S Ã O',
              'Propução',             'propulsão',             'pRoPuLsÃo',
             'Propulçao',             'Propulsam',             'PrOpUlSãO',
             'Ancoragem',             'AnCoRaGeM',             'Encoragem',
            'Ancoraguem',              'Ancorajm',             'AncorageM',
     'A N C O R A G E M',             'ANCORAGEM',             'aNcOrAgEm',
             'Ancorajem',              'Encoragi',             'ancoragem',
             'Ancorajen',             'AncorajeM',             'Ancoragen'

2.2 - Conversão de valores para o tipo numérico:

In [47]:
# ANTES DE CONVERTER, NECESSÁRIO RETIRAR CARACTERES DE CIFRÃO
df['price'] = df['price'].astype(str).str.replace('R$', '').str.strip()
df['price'] = pd.to_numeric(df['price'], errors='coerce')

#CONFERINDO RESULTADOS DA CONVERSÃO
print(f"Tipo da coluna price: {df['price'].dtype}")
print(f"Qtd dos valores convertidos: {df['price'].value_counts().sum()}")

Tipo da coluna price: float64
Qtd dos valores convertidos: 157


2.3 - Remoção de duplicados:

In [48]:
#REMOVENDO REGISTROS DUPLICADOS
antes_remocao = len(df)
df = df.drop_duplicates()
depois_remocao = len(df)
print(f"Registros removidos: {antes_remocao - depois_remocao}")
print(f"Dataset com {depois_remocao} linhas normalizadas e sem duplicatas.")


Registros removidos: 7
Dataset com 150 linhas normalizadas e sem duplicatas.


In [49]:
#SALVANDO O DATASET NORMALIZADO
df.to_csv('../data/processed/produtos_normalizados.csv', index=False)

#### 3 - CUSTOS DE IMPORTAÇÃO

In [50]:
#IMPORTANDO BIBLIOTECAS
import pandas as pd
import json

3.1 - Carregar o JSON e gerar novo arquivo CSV:

In [51]:
#CARREGANDO JSON
with open('../data/raw/custos_importacao.json', 'r', encoding='utf-8') as arquivo_json:
    dados = json.load(arquivo_json)

# DESANINHANDO JSON PARA CRIAR O DATAFRAME
df_custos_importacao = pd.json_normalize(dados,
                                         record_path=['historic_data'],
                                         meta=['product_id', 'product_name', 'category'])

df_custos_importacao = df_custos_importacao[[
    'product_id', 'product_name', 'category', 'start_date', 'usd_price']]

# EXIBINDO AS PRIMEIRAS LINHAS, AS ÚLTIMAS LINHAS E O FORMATO DO DATAFRAME
display(df_custos_importacao.head(2))
display(df_custos_importacao.tail(2))
print(df_custos_importacao.shape)

#AVALIANDO OS TIPOS DE DADOS
print(f"Tipos de dados:\n{df_custos_importacao.dtypes}")

,product_id,product_name,category,start_date,usd_price
0,1,Transponder AIS Maré Magnum,eletrônicos,10/08/2016,10583.63
1,1,Transponder AIS Maré Magnum,eletrônicos,15/06/2018,8778.36


,product_id,product_name,category,start_date,usd_price
1258,150,Cabo de Nylon Danforth Magnum Vox,ancoragem,04/10/2024,300.96
1259,150,Cabo de Nylon Danforth Magnum Vox,ancoragem,25/10/2024,288.90


(1260, 5)
Tipos de dados:
product_id       object
product_name        str
category            str
start_date          str
usd_price       float64
dtype: object


In [53]:
# CONVERTENDO TIPOS DE DADOS
df_custos_importacao['product_id'] = df_custos_importacao['product_id'].astype(
    int)  

df_custos_importacao['start_date'] = pd.to_datetime(df_custos_importacao['start_date'], 
                                                    errors='coerce', 
                                                    dayfirst=True)

df_custos_importacao['usd_price'] = pd.to_numeric(df_custos_importacao['usd_price'], 
                                                  errors='coerce')

print(f"\nTipos de dados convertidos:\n{df_custos_importacao.dtypes}")
display(df_custos_importacao.head(3))



Tipos de dados convertidos:
product_id               int64
product_name               str
category                   str
start_date      datetime64[us]
usd_price              float64
dtype: object


,product_id,product_name,category,start_date,usd_price
0,1,Transponder AIS Maré Magnum,eletrônicos,2016-08-10,10583.63
1,1,Transponder AIS Maré Magnum,eletrônicos,2018-06-15,8778.36
2,1,Transponder AIS Maré Magnum,eletrônicos,2018-09-25,8023.87


In [54]:
# SALVANDO EM CSV
df_custos_importacao.to_csv('../data/processed/custos_importacao.csv', index=False)